# Grover's Search Algorithm Implementation - Exercise Version

This notebook implements Grover's quantum search algorithm, which can search an unsorted database quadratically faster than classical algorithms. We'll demonstrate finding a specific 3-bit solution from 8 possible combinations.

**Your task:** Complete the missing code sections marked with `# TODO: FILL IN` comments to make the algorithm work correctly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit.visualization import plot_histogram
from qiskit.primitives import StatevectorSampler

In [ ]:
def simple_oracle_demo():
    """Demonstrate what our oracle should do classically"""
    
    test_cases = ['000', '001', '010', '011', '100', '101', '110', '111']
    
    print("Testing our oracle (the function that checks if answer is correct):")
    for bits in test_cases:
        x1, x2, x3 = [int(b) for b in bits]
        
        # Check our formula: (x1 OR x2 OR NOT x3) AND (NOT x1 OR NOT x2 OR NOT x3) AND (NOT x1 OR x2 OR x3)
        clause1 = x1 or x2 or (not x3)
        clause2 = (not x1) or (not x2) or (not x3)  
        clause3 = (not x1) or x2 or x3
        
        # Count how many variables are True in each clause
        clause1_count = sum([x1, x2, not x3])
        clause2_count = sum([not x1, not x2, not x3])
        clause3_count = sum([not x1, x2, x3])
        
        # We want exactly 1 True variable per clause
        is_solution = (clause1_count == 1 and clause2_count == 1 and clause3_count == 1)
        
        status = "✓ SOLUTION!" if is_solution else "✗"
        print(f"{bits}: {status}")

In [ ]:
def quantum_oracle(circuit, input_qubits, output_qubit, aux_qubits):
    """
    Build a quantum circuit that marks the correct solution (101).
    """
    
    # Step 1: Flip the third qubit (since we want NOT x3 in our solution)
    # TODO: FILL IN - Apply X gate to the third input qubit
    # Hint: We want to flip input_qubits[2] to check for NOT x3
    pass  # Replace with: circuit.x(input_qubits[2])
    
    # Step 2: Use controlled gates to flip output when all inputs are 1 (which represents "101")
    # TODO: FILL IN - Implement the three-qubit controlled operation
    # Hint: Use two CCX gates and an auxiliary qubit to implement 3-qubit control
    pass  # Replace with: circuit.ccx(input_qubits[0], input_qubits[1], aux_qubits[0])
    pass  # Replace with: circuit.ccx(input_qubits[2], aux_qubits[0], output_qubit)
    pass  # Replace with: circuit.ccx(input_qubits[0], input_qubits[1], aux_qubits[0])  # Uncompute
    
    # Step 3: Flip back the third qubit to restore original state
    # TODO: FILL IN - Apply X gate again to restore the original state
    pass  # Replace with: circuit.x(input_qubits[2])

In [ ]:
def amplify_solution(circuit, input_qubits):
    """
    The amplification step that increases probability of measuring the correct answer.
    This implements the inversion about average operation.
    """
    
    n = len(input_qubits)
    
    # Step 1: Apply H-gates to all qubits
    # TODO: FILL IN - Apply Hadamard gates to all input qubits
    for i in range(n):
        pass
    
    # Step 2: Apply X-gates to all qubits
    # TODO: FILL IN - Apply X gates to all input qubits
    for i in range(n):
        pass
    
    # Step 3: Apply controlled-Z gate (this flips the phase of |000⟩ state)
    # TODO: FILL IN - Implement controlled-Z using H-CCX-H pattern
    pass
    
    # Step 4: Apply X-gates again to flip back
    # TODO: FILL IN - Apply X gates again to flip back
    for i in range(n):
        pass

    # Step 5: Apply H-gates again
    # TODO: FILL IN - Apply Hadamard gates again
    for i in range(n):
        pass

In [ ]:
def run_grover_search(num_iterations=2):
    """
    Run the complete Grover search algorithm
    """
    
    # Set up quantum registers
    n = 3  # Number of variables (x1, x2, x3)
    input_qubits = QuantumRegister(n, name='input')
    output_qubit = QuantumRegister(1, name='output') 
    aux_qubits = QuantumRegister(2, name='helper')
    measurement = ClassicalRegister(n, name='result')
    
    # Create the quantum circuit
    circuit = QuantumCircuit()
    circuit.add_register(input_qubits)
    circuit.add_register(output_qubit)
    circuit.add_register(aux_qubits)
    circuit.add_register(measurement)
    
    # STEP 1: Initialize qubits in superposition
    print(f"\nStep 1: Creating superposition of all {2**n} possibilities...")
    for i in range(n):
        circuit.h(input_qubits[i])
    
    # Initialize output qubit in special state
    circuit.x(output_qubit[0])
    circuit.h(output_qubit[0])
    
    # STEP 2: Apply Grover iterations
    print(f"Step 2: Running {num_iterations} Grover iterations...")
    for iteration in range(num_iterations):
        print(f"  Iteration {iteration + 1}")
        
        # Apply oracle (mark the solution)
        quantum_oracle(circuit, input_qubits, output_qubit[0], aux_qubits)
        
        # Apply amplifier (boost probability of solution)
        amplify_solution(circuit, input_qubits)
    
    # STEP 3: Measure the result
    print("Step 3: Measuring the quantum state...")
    for i in range(n):
        circuit.measure(input_qubits[i], measurement[i])
    
    return circuit

In [ ]:
def analyze_results(circuit, shots=1024):
    """Run the quantum circuit and analyze results"""
    
    try:
        # Run on quantum simulator
        sampler = StatevectorSampler()
        job = sampler.run([circuit], shots=shots)
        result = job.result()
        counts = result[0].data.result.get_counts()
        
        print(f"\n=== RESULTS after {shots} measurements ===")
        
        # Sort results by frequency
        sorted_results = sorted(counts.items(), key=lambda x: x[1], reverse=True)
        
        for bitstring, count in sorted_results:
            probability = count / shots
            is_solution = bitstring == "101"
            marker = " ← CORRECT SOLUTION!" if is_solution else ""
            print(f"{bitstring}: {count:4d} times ({probability:.1%}){marker}")
        
        # Create visualization
        plt.figure(figsize=(10, 6))
        plot_histogram(counts, title="Grover Search Results")
        plt.tight_layout()
        plt.show()
        
        return counts
    
    except Exception as e:
        print(f"Error running circuit: {e}")
        return {}

In [ ]:
def compare_iterations():
    """See how the number of iterations affects success probability"""
    
    iteration_counts = [0, 1, 2, 3, 4]
    success_rates = []
    
    print("\n=== COMPARING DIFFERENT ITERATION COUNTS ===")
    
    for num_iter in iteration_counts:
        try:
            circuit = run_grover_search(num_iterations=num_iter)
            sampler = StatevectorSampler()
            job = sampler.run([circuit], shots=1000)
            result = job.result()
            counts = result[0].data.result.get_counts()
            
            # Calculate success rate
            success_count = counts.get("101", 0)
            success_rate = success_count / 1000
            success_rates.append(success_rate)
            
            print(f"Iterations: {num_iter}, Success Rate: {success_rate:.1%}")
            
        except Exception as e:
            print(f"Error with {num_iter} iterations: {e}")
            success_rates.append(0)
    
    # Plot the results
    if success_rates and max(success_rates) > 0:
        plt.figure(figsize=(10, 6))
        plt.plot(iteration_counts, success_rates, 'bo-', linewidth=2, markersize=8)
        plt.xlabel('Number of Grover Iterations')
        plt.ylabel('Success Probability')
        plt.title('How Iteration Count Affects Success Rate')
        plt.grid(True, alpha=0.3)
        plt.xticks(iteration_counts)
        
        # Add labels to points
        for i, rate in zip(iteration_counts, success_rates):
            plt.annotate(f'{rate:.1%}', (i, rate), textcoords="offset points", 
                        xytext=(0,10), ha='center')
        
        plt.tight_layout()
        plt.show()
        
        optimal_iterations = iteration_counts[success_rates.index(max(success_rates))]
        print(f"\nOptimal number of iterations: {optimal_iterations}")
        print(f"Maximum success rate: {max(success_rates):.1%}")

## Exercise Instructions

Before running Grover's algorithm, you need to complete the missing code sections in the functions above.

**What you need to fill in:**

### 1. quantum_oracle function:
- Apply X gate to flip the third qubit (for NOT x3)
- Implement 3-qubit controlled operation using CCX gates
- Restore the original state by applying X gate again

### 2. amplify_solution function:
- Apply Hadamard gates to all qubits
- Apply X gates to all qubits  
- Implement controlled-Z using H-CCX-H pattern
- Apply X gates again to flip back
- Apply Hadamard gates again

### 3. run_grover_search function:
- Apply Hadamard gates to create superposition
- Prepare output qubit in |-> state

**Available operations:**
- `circuit.h(qubit)` - Hadamard gate
- `circuit.x(qubit)` - X gate (NOT gate)
- `circuit.ccx(control1, control2, target)` - Toffoli gate (CCX)


In [ ]:
print("Understanding the Oracle")
print("=" * 40)
simple_oracle_demo()

## Understanding the Oracle

First, let's understand what our oracle function should identify as the correct solution:

In [ ]:
print("Running Grover Search (2 iterations)")
print("=" * 40)
circuit = run_grover_search(num_iterations=2)
results = analyze_results(circuit)

## Running Grover's Algorithm

Now let's run Grover's search algorithm with the optimal number of iterations (2 for our 3-qubit problem).

**Note:** Complete the missing code sections above before running this cell!

In [ ]:
compare_iterations()

## Experiment with Different Parameters

Try running the algorithm with different numbers of iterations:

In [ ]:
# Try with 1 iteration
print("Testing with 1 iteration:")
circuit_1 = run_grover_search(num_iterations=1)
results_1 = analyze_results(circuit_1)

In [ ]:
# Try with 3 iterations
print("Testing with 3 iterations:")
circuit_3 = run_grover_search(num_iterations=3)
results_3 = analyze_results(circuit_3)

## Complete Demonstration

Run the complete demonstration with summary analysis:

In [ ]:
def run_complete_grover_demo():
    """Run the complete Grover algorithm demonstration"""
    
    print("=" * 60)
    print("GROVER'S SEARCH ALGORITHM DEMONSTRATION")
    print("=" * 60)
    
    print("\nWelcome to Quantum Search!")
    print("We're looking for a 3-bit solution where each bit is 0 or 1")
    print("There are 2³ = 8 possible answers: 000, 001, 010, 011, 100, 101, 110, 111")
    
    # Step 1: Show what the oracle should do
    print("\n" + "="*40)
    print("STEP 1: Understanding the Oracle")
    print("="*40)
    simple_oracle_demo()
    
    # Step 2: Run Grover search with optimal iterations
    print("\n" + "="*40)
    print("STEP 2: Running Grover Search (2 iterations)")
    print("="*40)
    circuit = run_grover_search(num_iterations=2)
    results = analyze_results(circuit)
    
    # Step 3: Compare different iteration counts
    print("\n" + "="*40)
    print("STEP 3: Comparing Different Iteration Counts")
    print("="*40)
    compare_iterations()
    
    # Summary
    if results:
        solution_probability = results.get("101", 0) / 1024
        print(f"\n" + "="*40)
        print("SUMMARY")
        print("="*40)
        print(f"Target solution: 101")
        print(f"Success probability with 2 iterations: {solution_probability:.1%}")
        print(f"Classical search would need up to 8 evaluations")
        print(f"Grover's algorithm needs only ~√8 ≈ 2-3 iterations!")
        
        if solution_probability > 0.8:
            print("✅ Excellent! The algorithm is working perfectly!")
        elif solution_probability > 0.5:
            print("✅ Good! The algorithm is working well!")
        else:
            print("⚠️  The success rate is lower than expected. Check the implementation.")

# Run the complete demonstration
run_complete_grover_demo()